# Cloud Chamber Particle Track Classifier

This notebook detects and classifies particle tracks in cloud chamber videos using:
- **Image processing** with adaptive thresholding and skeletonisation
- **Feature extraction** (12 geometric features from track skeletons)
- **Random Forest classifier** for particle type prediction

### Particle Track Types
| Class | Label | Characteristics |
|-------|-------|----------------|
| 0 | Alpha particle | Thick, short, straight |
| 1 | Proton | Thin, long, straight |
| 2 | Electron/Positron/Muon | Thin, curved/wiggly |
| 3 | Low-energy electron | Tight spiral/curly |
| 4 | Knock-on electron | Forked track |

## Cell 1: Setup & Imports

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install opencv-python numpy scikit-image scikit-learn matplotlib joblib scipy

import os
import sys
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from skimage.morphology import skeletonize
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Import our local modules
import config
import track_detector
import feature_extractor
import classifier

# For inline plots
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 100

print("All imports successful!")
print(f"OpenCV version: {cv2.__version__}")
print(f"Images directory: {config.IMAGES_DIR}")
print(f"Number of annotated images: {len(glob.glob(os.path.join(config.IMAGES_DIR, '*.jpg')))}")

## Cell 2: Configuration & Paths

Review and optionally override configuration parameters.

In [ ]:
# Display current configuration
print("=" * 60)
print("PARTICLE CLASSES")
print("=" * 60)
for idx, label in config.PARTICLE_LABELS.items():
    print(f"  Class {idx}: {label}")

print(f"\n{'=' * 60}")
print("PREPROCESSING PARAMETERS")
print("=" * 60)
print(f"  CLAHE clip limit:     {config.CLAHE_CLIP_LIMIT}")
print(f"  Denoise strength (h): {config.DENOISE_H}")
print(f"  Adaptive block size:  {config.ADAPTIVE_BLOCK_SIZE}")
print(f"  Adaptive C:           {config.ADAPTIVE_C}")
print(f"  Min contour area:     {config.MIN_CONTOUR_AREA}")
print(f"  Max contour area:     {config.MAX_CONTOUR_AREA}")

print(f"\n{'=' * 60}")
print("RANDOM FOREST PARAMETERS")
print("=" * 60)
print(f"  n_estimators:     {config.RF_N_ESTIMATORS}")
print(f"  max_depth:        {config.RF_MAX_DEPTH}")
print(f"  min_samples_split: {config.RF_MIN_SAMPLES_SPLIT}")
print(f"  min_samples_leaf:  {config.RF_MIN_SAMPLES_LEAF}")

print(f"\n{'=' * 60}")
print(f"IMAGE-LABEL MAPPING ({len(config.IMAGE_LABEL_MAP)} entries)")
print("=" * 60)
total_boxes = 0
class_counts = defaultdict(int)
for key, labels in config.IMAGE_LABEL_MAP.items():
    short_key = key.split('snapshot_')[1] if 'snapshot_' in key else key
    label_names = [config.PARTICLE_LABELS[l] for l in labels]
    print(f"  {short_key}: {label_names}")
    total_boxes += len(labels)
    for l in labels:
        class_counts[l] += 1

print(f"\nTotal annotated boxes: {total_boxes}")
print("\nClass distribution in annotations:")
for cls_id in sorted(class_counts):
    print(f"  {config.PARTICLE_LABELS[cls_id]}: {class_counts[cls_id]}")

## Cell 3: Training Data Extraction

Load annotated images, detect red bounding boxes, and extract track skeletons with their labels.

In [ ]:
def match_image_to_labels(filename):
    """
    Match an image filename to its labels from the IMAGE_LABEL_MAP.
    Uses substring matching to handle varying filename formats.
    """
    for key, labels in config.IMAGE_LABEL_MAP.items():
        if key in filename:
            return labels
    return None


# Collect all annotated images
image_files = sorted(glob.glob(os.path.join(config.IMAGES_DIR, "*.jpg")))
print(f"Found {len(image_files)} annotated images\n")

# Extract training data
all_training_tracks = []  # List of (TrackCandidate, label)
matched_count = 0
unmatched_files = []

for img_path in image_files:
    filename = os.path.basename(img_path)
    labels = match_image_to_labels(filename)
    
    if labels is None:
        unmatched_files.append(filename)
        continue
    
    matched_count += 1
    tracks = track_detector.extract_training_data_from_image(img_path, labels)
    
    label_names = [config.PARTICLE_LABELS[l] for l in labels]
    print(f"  {filename[:60]}...")
    print(f"    Labels: {label_names}")
    print(f"    Red boxes found: {len(track_detector.detect_red_boxes(cv2.imread(img_path)))}")
    print(f"    Tracks extracted: {len(tracks)}")
    
    all_training_tracks.extend(tracks)

if unmatched_files:
    print(f"\nWarning: {len(unmatched_files)} image(s) had no label mapping:")
    for f in unmatched_files:
        print(f"  - {f}")

print(f"\n{'=' * 60}")
print(f"Total images matched: {matched_count}")
print(f"Total tracks extracted: {len(all_training_tracks)}")

# Count per class
extracted_class_counts = defaultdict(int)
for _, label in all_training_tracks:
    extracted_class_counts[label] += 1

print("\nExtracted tracks per class:")
for cls_id in sorted(extracted_class_counts):
    print(f"  {config.PARTICLE_LABELS[cls_id]}: {extracted_class_counts[cls_id]}")

In [ ]:
# Visualise some extracted training tracks
n_show = min(15, len(all_training_tracks))
if n_show > 0:
    cols = 5
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols * 2, figsize=(20, 4 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(n_show):
        track, label = all_training_tracks[i]
        row = i // cols
        col = (i % cols) * 2
        
        # Show binary mask
        axes[row, col].imshow(track.binary_mask, cmap='gray')
        axes[row, col].set_title(f"{config.PARTICLE_LABELS[label]}\n(binary)", fontsize=8)
        axes[row, col].axis('off')
        
        # Show skeleton
        axes[row, col + 1].imshow(track.skeleton, cmap='gray')
        axes[row, col + 1].set_title(f"{config.PARTICLE_LABELS[label]}\n(skeleton)", fontsize=8)
        axes[row, col + 1].axis('off')
    
    # Hide unused subplots
    for i in range(n_show, rows * cols):
        row = i // cols
        col = (i % cols) * 2
        if row < axes.shape[0] and col + 1 < axes.shape[1]:
            axes[row, col].axis('off')
            axes[row, col + 1].axis('off')
    
    plt.suptitle("Extracted Training Tracks: Binary Masks & Skeletons", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No tracks were extracted. Check preprocessing parameters.")

## Cell 4: Feature Extraction & Visualisation

Extract the 12 geometric features from each training track skeleton.

In [ ]:
# Extract features from all training tracks
X_real = []
y_real = []

for track, label in all_training_tracks:
    try:
        features = feature_extractor.extract_features(
            track.skeleton, track.binary_mask, track.bbox
        )
        # Check for NaN or infinite values
        if np.any(np.isnan(features)) or np.any(np.isinf(features)):
            continue
        X_real.append(features)
        y_real.append(label)
    except Exception as e:
        print(f"  Feature extraction failed for a {config.PARTICLE_LABELS[label]} track: {e}")

X_real = np.array(X_real)
y_real = np.array(y_real)

print(f"Feature matrix shape: {X_real.shape}")
print(f"Label vector shape: {y_real.shape}")
print(f"\nFeature names: {config.FEATURE_NAMES}")

# Display as a simple table
print(f"\n{'Feature':<25} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 65)
for i, name in enumerate(config.FEATURE_NAMES):
    if X_real.shape[0] > 0:
        col = X_real[:, i]
        print(f"{name:<25} {col.mean():>10.2f} {col.std():>10.2f} {col.min():>10.2f} {col.max():>10.2f}")

# Per-class feature statistics
print(f"\n{'=' * 60}")
print("PER-CLASS FEATURE MEANS")
print("=" * 60)
for cls_id in sorted(np.unique(y_real)):
    mask = y_real == cls_id
    if mask.sum() == 0:
        continue
    print(f"\n  {config.PARTICLE_LABELS[cls_id]} (n={mask.sum()}):")
    for i, name in enumerate(config.FEATURE_NAMES):
        col = X_real[mask, i]
        print(f"    {name:<25} mean={col.mean():.2f}  std={col.std():.2f}")

In [ ]:
# Visualise feature distributions per class (box plots)
if X_real.shape[0] > 0:
    n_features = len(config.FEATURE_NAMES)
    fig, axes = plt.subplots(3, 4, figsize=(20, 12))
    axes = axes.flatten()
    
    unique_classes = sorted(np.unique(y_real))
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#DDA0DD']
    
    for i, (name, ax) in enumerate(zip(config.FEATURE_NAMES, axes)):
        data_per_class = []
        labels_for_plot = []
        for cls_id in unique_classes:
            mask = y_real == cls_id
            if mask.sum() > 0:
                data_per_class.append(X_real[mask, i])
                labels_for_plot.append(config.PARTICLE_SHORT_LABELS[cls_id])
        
        bp = ax.boxplot(data_per_class, labels=labels_for_plot, patch_artist=True)
        for j, patch in enumerate(bp['boxes']):
            patch.set_facecolor(colors[j % len(colors)])
            patch.set_alpha(0.7)
        
        ax.set_title(name, fontsize=9, fontweight='bold')
        ax.tick_params(axis='x', rotation=45, labelsize=7)
    
    plt.suptitle("Feature Distributions by Particle Type", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No features to visualise.")

## Cell 5: Synthetic Data Augmentation

Generate synthetic training samples to supplement the limited real data.
Uses physics-informed Gaussian distributions centred on the observed feature
statistics from real tracks, with priors from domain knowledge.

In [ ]:
def generate_synthetic_features(X_real, y_real, n_per_class=200, random_state=42):
    """
    Generate synthetic feature vectors for each class.
    
    Strategy:
    1. If real samples exist for a class, compute mean/std and sample around them
    2. If no real samples exist, use physics-informed priors
    3. Add noise proportional to the feature's natural variance
    """
    rng = np.random.RandomState(random_state)
    
    # Physics-informed priors (fallback when no real data available)
    # Format: {class_id: {feature_index: (mean, std)}}
    # Based on known physics of cloud chamber tracks
    priors = {
        0: {  # Alpha: short, thick, straight
            0: (30, 15),    # skeleton_length: short
            1: (1.2, 0.5),  # bbox_aspect_ratio
            2: (8, 4),      # track_thickness: thick
            3: (0.85, 0.1), # straightness: very straight
            4: (0.5, 0.3),  # total_curvature: low
            5: (0.2, 0.1),  # max_curvature: low
            6: (0, 0.2),    # num_branch_points: 0
            7: (2, 0.5),    # num_endpoints: 2
            8: (0, 0.05),   # branch_ratio: 0
            9: (300, 150),  # contour_area
            10: (0.1, 0.05),# skeleton_area_ratio: low (thick)
            11: (0.4, 0.15),# bbox_fill_ratio
        },
        1: {  # Proton: long, thin, straight
            0: (80, 40),    # skeleton_length: long
            1: (0.4, 0.3),  # bbox_aspect_ratio
            2: (3, 1.5),    # track_thickness: thin
            3: (0.8, 0.12), # straightness: straight
            4: (1.0, 0.6),  # total_curvature: low
            5: (0.3, 0.15), # max_curvature: low
            6: (0, 0.3),    # num_branch_points: 0
            7: (2, 0.5),    # num_endpoints: 2
            8: (0, 0.05),   # branch_ratio: 0
            9: (250, 120),  # contour_area
            10: (0.3, 0.1), # skeleton_area_ratio: higher (thin)
            11: (0.2, 0.1), # bbox_fill_ratio
        },
        2: {  # Electron/Positron/Muon: curved, thin
            0: (60, 30),    # skeleton_length: medium
            1: (0.8, 0.4),  # bbox_aspect_ratio
            2: (3, 1.5),    # track_thickness: thin
            3: (0.5, 0.2),  # straightness: moderate curve
            4: (4, 2),      # total_curvature: moderate
            5: (0.6, 0.3),  # max_curvature: moderate
            6: (0, 0.5),    # num_branch_points: usually 0
            7: (2, 0.7),    # num_endpoints: 2
            8: (0, 0.08),   # branch_ratio: ~0
            9: (200, 100),  # contour_area
            10: (0.3, 0.1), # skeleton_area_ratio
            11: (0.15, 0.08),# bbox_fill_ratio
        },
        3: {  # Low-energy electron: spiral, short
            0: (40, 20),    # skeleton_length: short-medium
            1: (1.0, 0.4),  # bbox_aspect_ratio: ~square
            2: (3, 1.5),    # track_thickness
            3: (0.2, 0.12), # straightness: very curved
            4: (8, 4),      # total_curvature: high
            5: (1.2, 0.5),  # max_curvature: high
            6: (0, 0.5),    # num_branch_points
            7: (2, 1),      # num_endpoints
            8: (0, 0.1),    # branch_ratio
            9: (150, 80),   # contour_area
            10: (0.25, 0.1),# skeleton_area_ratio
            11: (0.15, 0.08),# bbox_fill_ratio
        },
        4: {  # Knock-on electron: forked
            0: (60, 30),    # skeleton_length
            1: (0.9, 0.4),  # bbox_aspect_ratio
            2: (4, 2),      # track_thickness
            3: (0.4, 0.15), # straightness
            4: (3, 2),      # total_curvature
            5: (0.8, 0.4),  # max_curvature
            6: (2, 1.5),    # num_branch_points: >=1 (KEY FEATURE)
            7: (4, 1.5),    # num_endpoints: >2 (KEY FEATURE)
            8: (0.3, 0.15), # branch_ratio: significant
            9: (250, 120),  # contour_area
            10: (0.25, 0.1),# skeleton_area_ratio
            11: (0.15, 0.08),# bbox_fill_ratio
        },
    }
    
    X_synthetic = []
    y_synthetic = []
    
    unique_classes = set(range(5))  # All 5 classes
    
    for cls_id in unique_classes:
        real_mask = y_real == cls_id
        n_real = real_mask.sum()
        
        for _ in range(n_per_class):
            sample = np.zeros(12)
            
            for feat_idx in range(12):
                if n_real >= 2:
                    # Use real data statistics with some added variance
                    real_mean = X_real[real_mask, feat_idx].mean()
                    real_std = max(X_real[real_mask, feat_idx].std(), 0.1)
                    # Blend with prior
                    prior_mean, prior_std = priors[cls_id][feat_idx]
                    # Weighted average: 70% real, 30% prior
                    mean = 0.7 * real_mean + 0.3 * prior_mean
                    std = 0.7 * real_std + 0.3 * prior_std
                elif n_real == 1:
                    # Single real sample: use it as mean with prior std
                    real_val = X_real[real_mask, feat_idx][0]
                    prior_mean, prior_std = priors[cls_id][feat_idx]
                    mean = 0.5 * real_val + 0.5 * prior_mean
                    std = prior_std
                else:
                    # No real data: use pure prior
                    mean, std = priors[cls_id][feat_idx]
                
                sample[feat_idx] = rng.normal(mean, std)
            
            # Enforce physical constraints
            sample[0] = max(sample[0], 5)       # skeleton_length >= 5
            sample[1] = max(sample[1], 0.05)    # aspect_ratio > 0
            sample[2] = max(sample[2], 0.5)     # thickness > 0
            sample[3] = np.clip(sample[3], 0, 1) # straightness in [0, 1]
            sample[4] = max(sample[4], 0)       # curvature >= 0
            sample[5] = max(sample[5], 0)       # max curvature >= 0
            sample[6] = max(sample[6], 0)       # branch points >= 0
            sample[7] = max(sample[7], 1)       # endpoints >= 1
            sample[8] = max(sample[8], 0)       # branch ratio >= 0
            sample[9] = max(sample[9], 10)      # area >= 10
            sample[10] = max(sample[10], 0.01)  # skeleton/area > 0
            sample[11] = np.clip(sample[11], 0, 1) # fill ratio in [0, 1]
            
            X_synthetic.append(sample)
            y_synthetic.append(cls_id)
    
    return np.array(X_synthetic), np.array(y_synthetic)


# Generate synthetic data
X_synth, y_synth = generate_synthetic_features(
    X_real, y_real,
    n_per_class=config.SYNTHETIC_SAMPLES_PER_CLASS,
)

# Combine real + synthetic
X_combined = np.vstack([X_real, X_synth]) if X_real.shape[0] > 0 else X_synth
y_combined = np.concatenate([y_real, y_synth]) if y_real.shape[0] > 0 else y_synth

print(f"Real samples:      {X_real.shape[0]}")
print(f"Synthetic samples: {X_synth.shape[0]}")
print(f"Combined total:    {X_combined.shape[0]}")
print(f"Feature dimensions: {X_combined.shape[1]}")

print("\nCombined class distribution:")
for cls_id in sorted(np.unique(y_combined)):
    n = (y_combined == cls_id).sum()
    n_real = (y_real == cls_id).sum() if y_real.shape[0] > 0 else 0
    print(f"  {config.PARTICLE_LABELS[cls_id]}: {n} ({n_real} real + {n - n_real} synthetic)")

## Cell 6: Model Training & Evaluation

Train the Random Forest classifier and evaluate its performance.

In [ ]:
# Train the model
model, cv_accuracy, cv_std, report = classifier.train_model(
    X_combined, y_combined, cv_folds=5
)

print(f"{'=' * 60}")
print(f"CROSS-VALIDATION RESULTS")
print(f"{'=' * 60}")
print(f"Accuracy: {cv_accuracy:.4f} (+/- {cv_std:.4f})")
print(f"\n{report}")

# Feature importance
print(f"{'=' * 60}")
print(f"FEATURE IMPORTANCE RANKING")
print(f"{'=' * 60}")
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
for rank, idx in enumerate(sorted_idx, 1):
    print(f"  {rank}. {config.FEATURE_NAMES[idx]:<25} {importances[idx]:.4f}")

In [ ]:
# Confusion matrix
cm, cm_labels = classifier.get_confusion_matrix(model, X_combined, y_combined)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cm_labels)
disp.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix (Training Data)', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Plot feature importance
colors_bar = plt.cm.viridis(np.linspace(0.3, 0.9, len(config.FEATURE_NAMES)))
bars = axes[1].barh(
    [config.FEATURE_NAMES[i] for i in sorted_idx],
    importances[sorted_idx],
    color=colors_bar[sorted_idx]
)
axes[1].set_xlabel('Importance')
axes[1].set_title('Feature Importance', fontsize=12, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Save the trained model
classifier.save_model(model)
print(f"\nModel saved to {config.MODEL_PATH}")

## Cell 7: Video Processing Pipeline

Process a cloud chamber video: detect tracks, classify them, and save an annotated output video.

In [ ]:
def process_video(input_path, output_path, model, 
                  skip_frames=1, max_frames=None, show_progress=True):
    """
    Process a cloud chamber video and output an annotated version.
    
    Parameters
    ----------
    input_path : str
        Path to the input video file.
    output_path : str
        Path for the output annotated video.
    model : RandomForestClassifier
        Trained classifier model.
    skip_frames : int
        Process every Nth frame (1 = every frame).
    max_frames : int or None
        Maximum number of frames to process (None = all).
    show_progress : bool
        Print progress updates.
    
    Returns
    -------
    stats : dict
        Processing statistics.
    """
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {input_path}")
    
    # Video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Video: {input_path}")
    print(f"  Resolution: {width}x{height}")
    print(f"  FPS: {fps:.1f}")
    print(f"  Total frames: {total_frames}")
    print(f"  Duration: {total_frames/fps:.1f}s")
    
    # Output writer
    fourcc = cv2.VideoWriter_fourcc(*config.VIDEO_CODEC)
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    if not out.isOpened():
        raise ValueError(f"Could not create output video: {output_path}")
    
    # Background subtractor
    bg_subtractor = cv2.createBackgroundSubtractorMOG2(
        history=config.BG_HISTORY,
        varThreshold=config.BG_THRESHOLD,
        detectShadows=config.BG_DETECT_SHADOWS,
    )
    
    # Statistics
    stats = {
        'total_frames': 0,
        'processed_frames': 0,
        'total_tracks': 0,
        'class_counts': defaultdict(int),
        'sample_frames': [],  # Store a few annotated frames for preview
    }
    
    frame_idx = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        stats['total_frames'] += 1
        frame_idx += 1
        
        if max_frames and stats['processed_frames'] >= max_frames:
            # Still write remaining frames unannotated
            out.write(frame)
            continue
        
        # Skip frames for faster processing
        if frame_idx % skip_frames != 0:
            out.write(frame)
            continue
        
        stats['processed_frames'] += 1
        annotated_frame = frame.copy()
        
        # Detect tracks using background subtraction
        tracks = track_detector.extract_tracks_from_frame(frame, bg_subtractor)
        
        # Classify each track and annotate
        for track in tracks:
            try:
                # Extract features
                features = feature_extractor.extract_features(
                    track.skeleton, track.binary_mask, track.bbox
                )
                
                if np.any(np.isnan(features)) or np.any(np.isinf(features)):
                    continue
                
                # Classify
                pred_class, probabilities = classifier.predict_track(model, features)
                confidence = probabilities.get(pred_class, 0.0)
                
                # Get display properties
                label = config.PARTICLE_SHORT_LABELS[pred_class]
                color = config.PARTICLE_COLORS[pred_class]
                x, y_pos, w, h = track.bbox
                
                # Draw bounding box
                cv2.rectangle(
                    annotated_frame,
                    (x, y_pos), (x + w, y_pos + h),
                    color, 2
                )
                
                # Draw label with confidence
                text = f"{label} ({confidence:.0%})"
                text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)[0]
                
                # Background rectangle for text readability
                cv2.rectangle(
                    annotated_frame,
                    (x, y_pos - text_size[1] - 8),
                    (x + text_size[0] + 4, y_pos),
                    color, -1  # Filled
                )
                cv2.putText(
                    annotated_frame, text,
                    (x + 2, y_pos - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    (255, 255, 255), 1, cv2.LINE_AA
                )
                
                stats['total_tracks'] += 1
                stats['class_counts'][pred_class] += 1
                
            except Exception as e:
                continue
        
        # Write annotated frame
        out.write(annotated_frame)
        
        # Save sample frames for preview (every 10% of video)
        if len(tracks) > 0 and len(stats['sample_frames']) < 8:
            if frame_idx % max(total_frames // 8, 1) == 0 or len(stats['sample_frames']) == 0:
                # Convert BGR to RGB for matplotlib
                stats['sample_frames'].append(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
        
        # Progress
        if show_progress and frame_idx % 100 == 0:
            pct = frame_idx / total_frames * 100 if total_frames > 0 else 0
            print(f"  Frame {frame_idx}/{total_frames} ({pct:.1f}%) — "
                  f"{stats['total_tracks']} tracks detected so far")
    
    cap.release()
    out.release()
    
    print(f"\n{'=' * 60}")
    print(f"PROCESSING COMPLETE")
    print(f"{'=' * 60}")
    print(f"Output saved to: {output_path}")
    print(f"Frames processed: {stats['processed_frames']}/{stats['total_frames']}")
    print(f"Total tracks detected: {stats['total_tracks']}")
    print(f"\nDetections per class:")
    for cls_id in sorted(stats['class_counts']):
        print(f"  {config.PARTICLE_LABELS[cls_id]}: {stats['class_counts'][cls_id]}")
    
    return stats

In [ ]:
# ============================================================
# SET YOUR VIDEO PATH HERE
# ============================================================
VIDEO_INPUT = r"path/to/your/cloud_chamber_video.mp4"
VIDEO_OUTPUT = os.path.join(config.BASE_DIR, "output_annotated.mp4")

# Optional: load a previously saved model instead of using the
# one trained above
# model = classifier.load_model()

# Process the video
if os.path.exists(VIDEO_INPUT):
    stats = process_video(
        input_path=VIDEO_INPUT,
        output_path=VIDEO_OUTPUT,
        model=model,
        skip_frames=1,      # Process every frame (set higher for speed)
        max_frames=None,    # Process all frames (set a number to limit)
    )
else:
    print(f"Video not found: {VIDEO_INPUT}")
    print("Please update VIDEO_INPUT with the correct path to your cloud chamber video.")
    stats = None

## Cell 8: Results Preview

Display sample annotated frames and detection statistics.

In [ ]:
if stats is not None and len(stats.get('sample_frames', [])) > 0:
    n_samples = len(stats['sample_frames'])
    cols = min(4, n_samples)
    rows = (n_samples + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 8 * rows))
    if n_samples == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes)
    
    for i, frame_rgb in enumerate(stats['sample_frames']):
        r, c = divmod(i, cols)
        axes[r, c].imshow(frame_rgb)
        axes[r, c].set_title(f'Sample Frame {i+1}', fontsize=10)
        axes[r, c].axis('off')
    
    # Hide unused subplots
    for i in range(n_samples, rows * cols):
        r, c = divmod(i, cols)
        if r < axes.shape[0] and c < axes.shape[1]:
            axes[r, c].axis('off')
    
    plt.suptitle('Sample Annotated Frames from Output Video', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Detection statistics chart
    if stats['total_tracks'] > 0:
        fig, ax = plt.subplots(figsize=(10, 5))
        class_names = [config.PARTICLE_LABELS[c] for c in sorted(stats['class_counts'])]
        counts = [stats['class_counts'][c] for c in sorted(stats['class_counts'])]
        colors_pie = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#DDA0DD']
        
        bars = ax.bar(class_names, counts, 
                      color=[colors_pie[c] for c in sorted(stats['class_counts'])])
        ax.set_ylabel('Number of Detections')
        ax.set_title('Particle Track Detection Distribution', 
                     fontsize=12, fontweight='bold')
        
        # Add count labels on bars
        for bar, count in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    str(count), ha='center', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
else:
    print("No video has been processed yet.")
    print("Set VIDEO_INPUT in the cell above and run it to process a video.")
    print("\nYou can also test the classifier on individual images:")

In [ ]:
# =============================================================
# BONUS: Test on a single image (useful for quick verification)
# =============================================================
def classify_single_image(image_path, model, show_steps=True):
    """
    Run the full pipeline on a single image and display results.
    """
    image = cv2.imread(image_path)
    if image is None:
        print(f"Could not load: {image_path}")
        return
    
    # Detect tracks (no background subtraction for single images)
    tracks = track_detector.extract_tracks_from_frame(image, bg_subtractor=None)
    print(f"Detected {len(tracks)} tracks in {os.path.basename(image_path)}")
    
    # Annotate
    annotated = image.copy()
    for track in tracks:
        try:
            features = feature_extractor.extract_features(
                track.skeleton, track.binary_mask, track.bbox
            )
            if np.any(np.isnan(features)) or np.any(np.isinf(features)):
                continue
            
            pred_class, probabilities = classifier.predict_track(model, features)
            confidence = probabilities.get(pred_class, 0.0)
            label = config.PARTICLE_SHORT_LABELS[pred_class]
            color = config.PARTICLE_COLORS[pred_class]
            x, y, w, h = track.bbox
            
            cv2.rectangle(annotated, (x, y), (x+w, y+h), color, 2)
            text = f"{label} ({confidence:.0%})"
            text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)[0]
            cv2.rectangle(annotated, (x, y-text_size[1]-8), (x+text_size[0]+4, y), color, -1)
            cv2.putText(annotated, text, (x+2, y-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)
            
            print(f"  Track at ({x},{y}): {config.PARTICLE_LABELS[pred_class]} ({confidence:.1%})")
        except Exception as e:
            print(f"  Error: {e}")
    
    if show_steps:
        # Show original vs annotated
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axes[0].set_title('Original')
        axes[0].axis('off')
        axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[1].set_title('Annotated')
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()
    
    return annotated


# Test on one of the annotated images
test_images = sorted(glob.glob(os.path.join(config.IMAGES_DIR, "*221024*.jpg")))
if test_images:
    # Pick an image with clear tracks
    test_img = test_images[min(7, len(test_images)-1)]  # 00.34 snapshot
    print(f"Testing on: {os.path.basename(test_img)}")
    _ = classify_single_image(test_img, model)
else:
    print("No test images found.")